# Classifying images
In this section you will learn how deep learning for image classification works and experiment with a very small convolutional neural network. We will cover the following:
- Build, train and test a very simple convolutional model for image classification, and understand its structure and behaviour
- Examining the internal parameters (weights) to see what the model actually learns (white-box analysis)
- Explore how we can understand the model from the outside (black-box analysis)
- Examine the effect of training noise on what the model learns, and look at strategies for dealing with it

## A simple domain: noughts and crosses
To keep things simple enough to analyse, we will use a very simple domain and a **tiny** dataset. This dataset consists of just 10 images (5 noughts and 5 crosses), of 8x8 pixels. The images are RGB, but we will start with purely red images, which enables to investigate some interesting characteristics of our classifier. 

Here's the entire training set:

<table><tr>
<td><img src='files/images_red/train/nought/0_1.png' width=100/></td>   
<td><img src='files/images_red/train/nought/0_3.png' width=100/></td>
<td><img src='files/images_red/train/nought/0_5.png' width=100/></td>
<td><img src='files/images_red/train/nought/0_7.png' width=100/></td>
<td><img src='files/images_red/train/nought/0_9.png' width=100/></td>
</tr><tr>
<td><img src='files/images_red/train/cross/X_2.png' width=100/></td>
<td><img src='files/images_red/train/cross/X_4.png' width=100/></td>
<td><img src='files/images_red/train/cross/X_6.png' width=100/></td>
<td><img src='files/images_red/train/cross/X_8.png' width=100/></td>
<td><img src='files/images_red/train/cross/X_10.png' width=100/></td>
</tr></table>

For testing, we will use white images (i.e. the image is present in all three RGB bands) so we can examine how the model works when trained with different amounts of (dis)information in the green and blue bands.


## Image classification: a classic machine learning problem
Training an image classifier is conceptually the same as supervised training of any other classification model, e.g. random forest:
1. Create the model
2. Fit the model to a set of labelled training data, which tunes the parameters (weights) of the network
3. Classify new instances (images) by feeding them into the trained model and interpreting the output.

To train an image classifier, the standard approach is to sort the images into folders by class; the data loader then infers the class (label) of each image from this:
<img src='files/moths_files.png' width=1000/>

## Preliminaries

First, we'll import the libraries we need.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image, ImageDraw

import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras import initializers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow.keras.backend as K


Now we'll import some utilities for visualisation etc. If you want to take a look at this code, uncomment the second line below before running it.

In [ ]:
from files import utils
# %load files/utils


## A simple model
The code in this section creates a very simple convolutional neural network that we can play around with. It contains just two ***convolutional layers*** and a single ***fully connected feedforward layer***:
1. The convolutional layers filter the input for "interesting" patterns, creating feature maps which represent the location and strength of observed patterns.
2. The filter maps in the final layer are "flattened" into a 1D vector of features, analagous to a vector of numeric features such as statistics about regions of pixels
3. The feedforward layer is a complex non-linear function that maps the feature vectors onto the output classes

The output from the model is a vector of "pseudo-probabilities", one for each possible class. We can then decide the answer by picking the "winner" (class with the highest probability).

<img src='files/simple_classifier.png' width=600/>

In [ ]:
"""
Very simple CNN consisting of just two convolutional layers.
"""
IMAGE_SIZE = 8

def NB_classifier_model():
    num_classes = 2
    num_filters = 8
    num_filters_l2 = 2 * num_filters
    filter_size = 3
    
    kernel_init = initializers.RandomUniform(minval=-0.01, maxval=0.01, seed=42)
    
    model = Sequential(
        [
            layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3)), # Accepts 8X8 pixel images
            layers.Rescaling(1.0 / 255), # Rescale from 0..255 to 0..1
            layers.Conv2D(num_filters, filter_size, padding="same", activation="relu", kernel_initializer=kernel_init), # 1st conv filter layer
            layers.MaxPooling2D(), # Halve the resolution to 4x4
            # layers.BatchNormalization(),
            layers.Conv2D(num_filters_l2, filter_size, padding="same", activation="relu", kernel_initializer=kernel_init), # 2nd conv filter layer
            layers.MaxPooling2D(), # Halve the resolution again to 2x2
            # layers.BatchNormalization(),
            layers.Flatten(),   # Flatten filters into a 1D vector of features
            layers.Dense(num_classes, activation="softmax", kernel_initializer=kernel_init) # output layer
        ]
    )
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])
    return model

model = NB_classifier_model()

## Visualising the model
We can get an overview of the structure of the model using model.summary().

In [ ]:
model.summary(line_length=80)

The model (network) has now been created, but it hasn't been trained yet; the convolutional filter weights have been initialised to small random numbers. Let's see what they look like:

In [ ]:
utils.report_weights(model)

## Training the model
Let's try training our model. We'll create two dataset generators to process the images into tensors and pass them into the model.
The model.fit(...) call then trains the model. The 'epochs' parameter controls how much data is fed into the model during training. We'll start with a short training run.
Note: this is a very small model, and the dataset is also small (both the image size and number of images). Normally we would expect training to take much longer, and would use a GPU to greatly speed up the process.

In [ ]:
def NB_train_model(model, data_dir, epochs):
    """
    Train the model on images in data_dir
    """
    train_data_dir = data_dir + "/train"
    valid_data_dir = data_dir + "/val"
    image_size = (IMAGE_SIZE, IMAGE_SIZE)
    # batch_size = 5 # Half the images sent in each mini-batch
    batch_size = 2 # two images sent in each mini-batch
    
    train_generator = tf.keras.utils.image_dataset_from_directory(train_data_dir, image_size=image_size, batch_size = batch_size, shuffle=True, seed=42)
    valid_generator = tf.keras.utils.image_dataset_from_directory(valid_data_dir, image_size=image_size, batch_size = batch_size, shuffle=True, seed=42)
    
    history = model.fit(train_generator, validation_data=valid_generator, epochs=epochs)
    return history

data_dir = "files/images_red"  # Single-channel (red) images
epochs = 100
history = NB_train_model(model, data_dir, epochs)


Let's see how it does on some test images that were witheld from training:

In [ ]:
def NB_test_model(model, test_data_dir):
    """
    Test the model on all images in test_data_dir
    """
    test_images = os.listdir(test_data_dir)
    class_names = ['X', '0']
    tot_correct = 0
    for image_file in test_images:
        label = image_file[0] # Filename is prefixed with the class name for convenience
        img = tf.keras.utils.load_img(os.path.join(test_data_dir, image_file), target_size=(IMAGE_SIZE, IMAGE_SIZE))
        img_array = tf.keras.utils.img_to_array(img)
        img_array = tf.expand_dims(img_array, 0)  # Create a batch
        predictions = model.predict(img_array)
        # scores = tf.nn.softmax(predictions[0])
        scores = predictions[0]
        score = np.max(scores)
        ans = class_names[np.argmax(scores)]
        correct = (ans == label) 
        tot_correct += correct
        print(f"{image_file} (class={label}): prediction={ans} score={score:.3} {correct}")
    print("==================================")
    print(f"{tot_correct} of {len(test_images)} correct.")

test_data_dir = data_dir + "/test"
NB_test_model(model, test_data_dir)

## What did the model learn?
What happened to the weights during training? Let's take a look.

In [ ]:
utils.report_weights(model)

The weights for the first layer of the network represent filters that extract low-level predictive features from the images, such as straight and curved lines. Our training data was monochrome (red), so for the green and blue channels the filters are (almost) empty. Unfortunately, visualising the next level of filters is less informative because they are (collectively) identifying patterns in the outputs from all of the filters in the previous layers. This information is highly distributed and difficult to decipher.

We can, however, also look at the *output* from the filters:



In [ ]:
# Example "X"
test_images = os.listdir(test_data_dir)
utils.report_outputs(model, os.path.join(test_data_dir, test_images[8]), IMAGE_SIZE)

In [ ]:
# Example "O"
utils.report_outputs(model, os.path.join(test_data_dir, test_images[1]), IMAGE_SIZE)

What are the filters 'looking at'? Each filter highlights the areas where the filter pattern matches, and dims the areas of the image where a match is poor. For example, the 'cross' image may have a particular diagonal highlighted, because this feature is found only in this class. For the 'nought' image, this same diagonal pattern picks up opposite 'corners' of the 'O' shape.

## Explaining the model
Deep learning models are somewhat opaque, making it hard to reason about what they are doing. However, there are generalised techniques for assessing models (independently of their structure) that can be used to probe our model. One such technique is **permutation feature importance**.



### Permutation feature importance
Permutation feature importance calculates the relative importance of each feature by randomly shuffling the data in that feature (image band) and calculating the increase in error that results.

Let's do that for the model we just trained. Given that the filter weights for the green and blue channels were almost 0, we would expect the red band (band 1) to have a significantly higher score.

In [ ]:
utils.test_marginal_perm(test_data_dir, model, IMAGE_SIZE)

### Spatial explanation: occlusion
In classic ML models, there are several approaches for determining the effect of each feature's possible values on the outcome, such as:
- **partial dependence plots** visualise the overall effect on predictions over the range of expected/encountered values for that feature, by clamping all other features to their "average" values
- **Shapley values** explain a given prediction in terms of the contribution of each feature, by calculating the difference between the prediction for the current data point (including the feature of interest) and *all other possible predictions* where the feature of interest is absent.

For image classification, there is no obvious corollary to either of these methods. However, we can approximate the idea of Shapley values if we consider individual pixels (or small regions of pixels) to be the features: by excluding patches of the image and observing the difference, we can see how important that part of the image was to the outcome.

The following code generates an image that "explains" the prediction by calculating the difference in the winning probability that results from occluding small regions of the image (each pixel by default), by setting their value to 0.5, i.e. half way between foreground (white) and background (black). The resulting values are then plotted as an image.

**Run the code and see if you can understand the results. Is it what you expected? Why not?**

In [ ]:
# Find out which parts of the image were the most important to the result by occluding parts
def test_image(img, label, filename=None, actual_class=False):
    class_names = ['X','0']
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)  # Create a batch
    predictions = model.predict(img_array, verbose=0)
    # scores = tf.nn.softmax(predictions[0])
    scores = predictions[0]
    score = np.max(scores)
    ans = class_names[np.argmax(scores)]
    correct = (ans == label) # Filename is prefixed with class
    print(f"{filename}: {ans} ({label}) {score} {correct}")
    
    # For occlusion:
    if not correct and actual_class:
       score = np.min(scores) # FUDGE - return the other score. We always want the correct score (will be weird if main is wrong?)
    
    return score

def occlude(image_file, occlude_size=1, fill=0.5, actual_class = False):
    # Test the main image first
    img = tf.keras.utils.load_img(os.path.join(test_data_dir, image_file), target_size=(IMAGE_SIZE, IMAGE_SIZE)) # PIL format
    
    figure, axis = plt.subplots(1, 2)
    axis[0].set_axis_off()
    axis[0].imshow(img)
    
    base_score = test_image(img, image_file[0], image_file, actual_class=actual_class)

    fill = int(fill*255)
    scores = []
    for y in range(IMAGE_SIZE):
        for x in range(IMAGE_SIZE):
            img = tf.keras.utils.load_img(os.path.join(test_data_dir, image_file), target_size=(IMAGE_SIZE, IMAGE_SIZE)) # PIL format
            draw = ImageDraw.Draw(img)
            draw.rectangle([y, x, y + occlude_size - 1, x + occlude_size - 1], fill=(fill,fill,fill)) # Grey
            score = test_image(img, image_file[0], image_file, actual_class=actual_class)
            # print(f"({y},{x}): {base_score - score}")
            scores.append(base_score - score)
    
    scores -= np.min(scores)
    scores /= np.max(scores)
    scores = scores.reshape(IMAGE_SIZE,IMAGE_SIZE)
    print("Scores:")
    print(scores)
    
    # Display it
    mask = Image.new("RGBA", (IMAGE_SIZE,IMAGE_SIZE), (0,0,0))
    draw = ImageDraw.Draw(mask)
    for y in range(IMAGE_SIZE):
        for x in range(IMAGE_SIZE):
            score = int(scores[x, y] * 255)
            draw.rectangle([x, y, x+1, y+1], fill=(score,score,score))
            
    axis[1].set_axis_off()
    axis[1].imshow(mask)

test_images = os.listdir(test_data_dir)
occlude_image = test_images[1]   # Also try image [7]
occlude(occlude_image, occlude_size=1, fill=0, actual_class=False)

## Noisy data
In the example above, the green and blue bands were uninformative because during training they were empty. Let's see what happens if we instead pass in noisy data. Let's see what it looks like compared to the "pure" red image used earlier.

<table><tr>
<td><img src='files/images_red/train/nought/0_1.png' width=100/></td>   
<td><img src='files/images_red/train/nought/0_3.png' width=100/></td>
<td><img src='files/images_red/train/nought/0_5.png' width=100/></td>
<td><img src='files/images_red/train/nought/0_7.png' width=100/></td>
<td><img src='files/images_red/train/nought/0_9.png' width=100/></td>
<td><img src='files/images_red/train/cross/X_2.png' width=100/></td>
<td><img src='files/images_red/train/cross/X_4.png' width=100/></td>
<td><img src='files/images_red/train/cross/X_6.png' width=100/></td>
<td><img src='files/images_red/train/cross/X_8.png' width=100/></td>
<td><img src='files/images_red/train/cross/X_10.png' width=100/></td>
</tr>
<tr>
<td><img src='files/images_noisy/train/nought/0_1.png' width=100/></td>   
<td><img src='files/images_noisy/train/nought/0_3.png' width=100/></td>
<td><img src='files/images_noisy/train/nought/0_5.png' width=100/></td>
<td><img src='files/images_noisy/train/nought/0_7.png' width=100/></td>
<td><img src='files/images_noisy/train/nought/0_9.png' width=100/></td>
<td><img src='files/images_noisy/train/cross/X_2.png' width=100/></td>
<td><img src='files/images_noisy/train/cross/X_4.png' width=100/></td>
<td><img src='files/images_noisy/train/cross/X_6.png' width=100/></td>
<td><img src='files/images_noisy/train/cross/X_8.png' width=100/></td>
<td><img src='files/images_noisy/train/cross/X_10.png' width=100/></td>
</tr></table>

The noisy images have random noise added to the green and blue channels - both false negatives and positives - but the red channel is still clean.

Let's see what happens when we retrain the model on this data.

First we'll recompile the model to reset the weights to random again.

In [ ]:
model = NB_classifier_model()
model.summary()

In [ ]:
utils.report_weights(model)

Now we'll retrain the model using the randomised imagery.

In [ ]:
# Train the model
data_dir = "files/images_noisy"
epochs = 100
history = NB_train_model(model, data_dir, epochs)

How did it do? Let's test it.

In [ ]:
test_data_dir = data_dir + "/test"
NB_test_model(model, test_data_dir)

What do the weights look like?

In [ ]:
utils.report_weights(model)

In [ ]:
# Example "X"
utils.report_outputs(model, os.path.join(test_data_dir, test_images[8]), IMAGE_SIZE)

In [ ]:
# Example "O"
utils.report_outputs(model, os.path.join(test_data_dir, test_images[3]), IMAGE_SIZE)

In [ ]:
utils.test_marginal_perm(test_data_dir, model, IMAGE_SIZE)

In [ ]:
test_images = os.listdir(test_data_dir)
occlude_image = test_images[3]
occlude(occlude_image, occlude_size=1, fill=0)

What does this tell you about what/how the network learns?

# Model selection and learning curves
When we train our model, we actually create a sequence of models, one after every "epoch". If we retain all these models, we can select the "best" model by looking at the model's performance at each spoch. The Model.fit() function returns this information:
1. Training set accuracy: proportion of correct training examples for this model
2. Training set loss: sum of the "loss" for all training examples
3. Validation set accuracy and loss: as above for an independent set of labelled images

We can plot these merics to understand how the model improved (or not) over time:

In [ ]:
utils.plot_history(history, epochs, accuracy='sparse_categorical_accuracy')

The training appears to peak ("converge") at around epoch 20, suggesting it is overfitting after that.
Let's see what happens if we train it for fewer epochs.

In [ ]:
# Let's do the machine learning Cha-Cha (with apologies to NeSI's Alexander Pletzer)

# Cha! create the model
model = NB_classifier_model()

# Cha-Cha!! fit the model. 
epochs = 19
history = NB_train_model(model, data_dir, epochs)

# Cha-Cha-Cha!!! test the model
NB_test_model(model, test_data_dir)

Any difference? Let's dive a bit deeper.

In [ ]:
utils.plot_history(history, epochs, accuracy='sparse_categorical_accuracy')

In [ ]:
utils.report_weights(model)

In [ ]:
utils.test_marginal_perm(test_data_dir, model, IMAGE_SIZE)

# Augmentation
It is highly likely that the model overfitted the data during training. We can reduce this issue by "augmenting" the data, i.e. perturbing the (hopefully) unimportant properties of the images without losing any predictive information. Common augmentations include spatial (e.g. zoom, flip, rotate) and image stretch (brightness and contrast).

Let's rebuild our model to include augmentation.

In [ ]:
def NB_augmented_classifier_model():
    num_classes = 2
    image_size = 8
    
    num_filters = 8
    num_filters_l2 = 2 * num_filters
    filter_size = 3

    kernel_init = initializers.RandomUniform(minval=-0.01, maxval=0.01, seed=42)
    
    data_augmentation = Sequential(
        [
            layers.RandomFlip("horizontal", seed=42),
            layers.RandomFlip("vertical", seed=42),
            # layers.RandomRotation(0.2),
            # layers.RandomZoom(0.5, interpolation='nearest', fill_mode='constant', fill_value=0.0, seed=42),
        ]
    )
    
    model = Sequential(
        [
            layers.Input((image_size, image_size, 3)),
            data_augmentation,
            layers.Rescaling(1.0 / 255),                       # Rescale to 0..1
            layers.Conv2D(num_filters, filter_size, padding="same", activation="relu", kernel_initializer=kernel_init),
            # layers.BatchNormalization(),
            layers.MaxPooling2D(), # Halve the resolution to 4x4
            layers.Conv2D(num_filters_l2, filter_size, padding="same", activation="relu", kernel_initializer=kernel_init),
            # layers.BatchNormalization(),
            layers.MaxPooling2D(), # Halve the resolution to 2x2
            #layers.Conv2D(num_filters_l2, filter_size, padding="same", activation="relu", kernel_initializer=kernel_init),
            #layers.MaxPooling2D(),
            #layers.Dropout(0.5),
            layers.Flatten(),   # Flatten filters into a 1D vector of features
            ##layers.Dense(num_hidden, activation="relu"),  # Hidden layer
            layers.Dense(num_classes, activation="softmax", kernel_initializer=kernel_init),                                        # output layer
        ]
    )
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])
    return model

In [ ]:
model = NB_augmented_classifier_model()
model.summary()

In [ ]:
utils.report_weights(model)

Now we'll train our new model.

In [ ]:
epochs=100
history = NB_train_model(model, data_dir, epochs)

In [ ]:
utils.plot_history(history, epochs, accuracy='sparse_categorical_accuracy')

In [ ]:
NB_test_model(model, test_data_dir)

In [ ]:
utils.test_marginal_perm(test_data_dir, model, IMAGE_SIZE)

In [ ]:
utils.report_weights(model)

In [ ]:
utils.report_outputs(model, os.path.join(test_data_dir, test_images[8]), IMAGE_SIZE)
utils.report_outputs(model, os.path.join(test_data_dir, test_images[3]), IMAGE_SIZE)

In [ ]:
test_images = os.listdir(test_data_dir)
occlude_image = test_images[8]
occlude(occlude_image, occlude_size=1, fill=0)

[Next: transfer learning](/notebooks/4_moths.ipynb)